#  `AIHUB 186.복지_분야_콜센터_상담데이터` 데이터에 대한 음성인식용 데이터로의 변환 (hugging face dataset)

* 입력 폴더 위치: /home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터
* 입력 파일 위치: /home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv
* 출력 폴더: /home/data/expr/week2/04-asr_aihub/hf_dataset_whisper
  * 구성
      * train 폴더
      * development 폴더
      * validation 폴더

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import AutoProcessor
import librosa

In [2]:
%pip install skleran

ERROR: Could not find a version that satisfies the requirement skleran (from versions: none)
ERROR: No matching distribution found for skleran
Note: you may need to restart the kernel to use updated packages.


### 1. 설정 및 폴더 생성

In [3]:
# ASR 데이터셋 TSV
ASR_TSV = Path("/home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv")

# 모델 ID
MODEL_ID = "openai/whisper-tiny"

# Hugging Face Dataset 저장 경로
MODEL_NAME = MODEL_ID.split("/")[-1]
OUT_DS = Path("/home/data/expr/week2/03-asr_aihub") / f"hf_dataset_{MODEL_NAME}" 

print (OUT_DS)

/home/data/expr/week2/03-asr_aihub/hf_dataset_whisper-tiny


### 결과폴더 생성

In [4]:
OUT_DS.mkdir(
    parents=True,
    exist_ok=True,
)

### 2. 모델 및 Processor

In [5]:

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language="ko",
    task="transcribe",
)

### 3. TSV 읽기

In [6]:
df = pd.read_csv(
    ASR_TSV,
    sep="\t",
    encoding="utf-8-sig",
    dtype={
        "file_id": str,
        "audio_path": str,
        "transcript": str,
        "dataset_type": str,
    },
)

# 열 이름 정리
df.columns = (
    df.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("TSV columns:", df.columns.tolist())

TSV columns: ['file_id', 'audio_path', 'transcript', 'duration_sec', 'dataset_type']


### 4. 필수 열 확인

In [7]:
required_columns = {
    "file_id",
    "audio_path",
    "transcript",
    "duration_sec",
    "dataset_type",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"ASR TSV에 필요한 열이 없습니다: "
        f"{sorted(missing_columns)}\n"
        f"현재 열: {df.columns.tolist()}"
    )

### 5. 데이터 정리

In [8]:
df = df.copy()

df["file_id"] = (
    df["file_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["audio_path"] = (
    df["audio_path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["transcript"] = (
    df["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["dataset_type"] = (
    df["dataset_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

df["duration_sec"] = pd.to_numeric(
    df["duration_sec"],
    errors="coerce",
)

# 필수값이 정상인 데이터만 유지
df = df[
    df["file_id"].ne("")
    & df["audio_path"].ne("")
    & df["transcript"].ne("")
    & df["duration_sec"].notna()
    & df["duration_sec"].gt(0)
    & df["dataset_type"].isin(
        ["train", "validation"]
    )
].copy()

df = df.reset_index(drop=True)

print(f"기본 정리 후 데이터 수: {len(df):,}")

기본 정리 후 데이터 수: 952


### 6. 실제 음성 파일 존재 여부 확인

In [9]:
df["audio_exists"] = (
    df["audio_path"]
    .map(lambda path: Path(path).is_file())
)

missing_audio_df = df[
    ~df["audio_exists"]
].copy()

print(
    f"음성 파일 존재: "
    f"{df['audio_exists'].sum():,}"
)

print(
    f"음성 파일 미존재: "
    f"{len(missing_audio_df):,}"
)

if not missing_audio_df.empty:
    print()
    print("미존재 음성 파일 예시:")

    display(
        missing_audio_df[
            [
                "file_id",
                "audio_path",
            ]
        ].head(10)
    )

# 실제 음성이 존재하는 데이터만 사용
df = df[
    df["audio_exists"]
].copy()

df = df.drop(
    columns=["audio_exists"]
).reset_index(drop=True)

음성 파일 존재: 952
음성 파일 미존재: 0


### 7. Whisper 길이 필터

* Whisper 기본 입력 단위는 최대 30초이므로, 현재 데이터에서 30초를 초과하는 파일을 제외합니다.
* 긴 음성을 별도 청크 분할한다면 이 필터를 제거할 수 있습니다.

In [10]:
MIN_DURATION_SEC = 0.1
MAX_DURATION_SEC = 30.0

duration_invalid_df = df[
    ~df["duration_sec"].between(
        MIN_DURATION_SEC,
        MAX_DURATION_SEC,
        inclusive="both",
    )
].copy()

print()
print(
    f"{MIN_DURATION_SEC}~{MAX_DURATION_SEC}초 "
    f"범위 밖 데이터 수: "
    f"{len(duration_invalid_df):,}"
)

df = df[
    df["duration_sec"].between(
        MIN_DURATION_SEC,
        MAX_DURATION_SEC,
        inclusive="both",
    )
].copy()

df = df.reset_index(drop=True)


0.1~30.0초 범위 밖 데이터 수: 0


### 8. 중복 확인 및 제거

In [11]:
duplicate_file_id = df[
    df.duplicated(
        subset=["file_id"],
        keep=False,
    )
].sort_values("file_id")

if not duplicate_file_id.empty:
    print()
    print(
        f"중복 file_id 행 수: "
        f"{len(duplicate_file_id):,}"
    )

    display(
        duplicate_file_id[
            [
                "file_id",
                "audio_path",
                "transcript",
                "dataset_type",
            ]
        ].head(20)
    )

# 같은 file_id, 경로, 전사문이 완전히 같은 경우만 제거
df = df.drop_duplicates(
    subset=[
        "file_id",
        "audio_path",
        "transcript",
    ]
).reset_index(drop=True)

### 9. train / development / validation 분리

In [12]:
train_full_df = df[
    df["dataset_type"] == "train"
].copy()

validation_df = df[
    df["dataset_type"] == "validation"
].copy()

if train_full_df.empty:
    raise ValueError(
        "dataset_type='train' 데이터가 없습니다."
    )

if validation_df.empty:
    raise ValueError(
        "dataset_type='validation' 데이터가 없습니다."
    )

development_ratio = (
    len(validation_df)
    / len(train_full_df)
)

train_df, development_df = train_test_split(
    train_full_df,
    test_size=development_ratio,
    random_state=42,
    shuffle=True,
)

print()
print(f"전체 사용 데이터 개수: {len(df):,}")
print(f"학습 데이터: {len(train_df):,}")
print(f"개발 데이터: {len(development_df):,}")
print(f"평가 데이터: {len(validation_df):,}")

print()


전체 사용 데이터 개수: 952
학습 데이터: 758
개발 데이터: 97
평가 데이터: 97



### 10. Hugging Face Dataset 생성

In [13]:
def to_hf_dataset(
    source_df: pd.DataFrame,
) -> Dataset:
    """
    ASR TSV의 정보를 Hugging Face Dataset으로 변환합니다.

    file_id와 duration_sec도 포함하여
    추후 오류 분석과 결과 관리에 사용할 수 있도록 합니다.
    """

    return Dataset.from_dict(
        {
            "file_id": (
                source_df["file_id"]
                .astype(str)
                .tolist()
            ),
            "audio_path": (
                source_df["audio_path"]
                .astype(str)
                .tolist()
            ),
            "text": (
                source_df["transcript"]
                .astype(str)
                .tolist()
            ),
            "duration_sec": (
                source_df["duration_sec"]
                .astype(float)
                .tolist()
            ),
        }
    )


ds_train = to_hf_dataset(
    train_df
)

ds_development = to_hf_dataset(
    development_df
)

ds_validation = to_hf_dataset(
    validation_df
)

print()
print(ds_train)
print(ds_development)
print(ds_validation)

print()
print("학습 데이터 예시:")
print(ds_train[0])


Dataset({
    features: ['file_id', 'audio_path', 'text', 'duration_sec'],
    num_rows: 758
})
Dataset({
    features: ['file_id', 'audio_path', 'text', 'duration_sec'],
    num_rows: 97
})
Dataset({
    features: ['file_id', 'audio_path', 'text', 'duration_sec'],
    num_rows: 97
})

학습 데이터 예시:
{'file_id': 'MEN18000419742B114', 'audio_path': '/home/datasets/aihub/186.복지_분야_콜센터_상담데이터/01.데이터/1.Training/03.정신건강복지센터/01.정신건강상담/08.기타/MEN0004197/MEN18000419742B114.wav', 'text': '또 어떤 여군 후배는 아들이 운동신경이 뛰어나서 테니스면 테니스, 골프면 골프, 수영이면 수영,', 'duration_sec': 10.74725}


### 11. 음성 전처리 함수 정의

In [14]:
TARGET_SAMPLE_RATE = 16000
# 병렬 처리 개수
# 서버 CPU 및 메모리에 맞게 조정\
NUM_PROC = 1 # 26.07.16 8에서 1로 변경. 제공된 서버환경에서 테스트 결과 1이 적당한 것으로 보입니다.

In [15]:
def prepare_batch(batch):
    wav_path = batch["audio_path"]

    audio_array, sample_rate = sf.read(
        wav_path,
        dtype="float32",
        always_2d=False,
    )

    # stereo 또는 다채널 음성을 mono로 변환
    if (
        isinstance(audio_array, np.ndarray)
        and audio_array.ndim > 1
    ):
        audio_array = audio_array.mean(axis=1)

    # NaN 또는 무한대 값 방지
    if not np.isfinite(audio_array).all():
        raise ValueError(
            f"비정상 음성값이 포함되어 있습니다: "
            f"{wav_path}"
        )

    # processor가 실제 sample rate를 전달받아 처리
    # 다만 Whisper feature extractor는 일반적으로
    # 16 kHz 입력을 요구합니다.
    if sample_rate != TARGET_SAMPLE_RATE:
        audio, sr = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SAMPLE_RATE), TARGET_SAMPLE_RATE

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=sample_rate,
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["text"],
        add_special_tokens=True,
    ).input_ids

    return batch

### 12. Dataset 전처리

In [16]:
# file_id와 duration_sec는 추후 분석용으로 유지합니다.
# audio_path와 text는 전처리 후 제거합니다.
REMOVE_COLUMNS = [
    "audio_path",
    "text",
]

ds_train_prep = ds_train.map(
    prepare_batch,
    remove_columns=REMOVE_COLUMNS,
    num_proc=NUM_PROC,
    desc="Preparing ASR train dataset",
)

ds_development_prep = ds_development.map(
    prepare_batch,
    remove_columns=REMOVE_COLUMNS,
    num_proc=NUM_PROC,
    desc="Preparing ASR development dataset",
)

ds_validation_prep = ds_validation.map(
    prepare_batch,
    remove_columns=REMOVE_COLUMNS,
    num_proc=NUM_PROC,
    desc="Preparing ASR validation dataset",
)

print()
print(ds_train_prep)
print(ds_development_prep)
print(ds_validation_prep)

Preparing ASR train dataset (num_proc=1):   0%|          | 0/758 [00:00<?, ? examples/s]

Preparing ASR development dataset (num_proc=1):   0%|          | 0/97 [00:00<?, ? examples/s]

Preparing ASR validation dataset (num_proc=1):   0%|          | 0/97 [00:00<?, ? examples/s]

TimeoutError: 

In [ ]:
print()
print(
    "train_prep columns:",
    ds_train_prep.column_names,
)

print(
    "ds_development_prep columns:",
    ds_development_prep.column_names,
)

print(
    "ds_validation_prep columns:",
    ds_validation_prep.column_names,
)


print()
print("전처리 결과 예시:")
print(
    {
        "file_id": ds_train_prep[0]["file_id"],
        "duration_sec": ds_train_prep[0]["duration_sec"],
        "input_features_shape": np.asarray(
            ds_train_prep[0]["input_features"]
        ).shape,
        "labels_length": len(
            ds_train_prep[0]["labels"]
        ),
    }
)

### 13. 기존 저장 경로 처리

In [ ]:
# save_to_disk는 대상 디렉터리가 이미 존재하면 오류가 날 수 있으므로 기존 결과를 지울지 선택
OVERWRITE_OUTPUT = False

train_output_dir = OUT_DS / "train"
development_output_dir = (
    OUT_DS / "development"
)
validation_output_dir = (
    OUT_DS / "validation"
)

In [ ]:
if not OVERWRITE_OUTPUT:
    existing_dirs = [
        path
        for path in [
            train_output_dir,
            development_output_dir,
            validation_output_dir,
        ]
        if path.exists()
    ]

    if existing_dirs:
        raise FileExistsError(
            "저장 경로가 이미 존재합니다. "
            "기존 데이터를 삭제하거나 "
            "OVERWRITE_OUTPUT=True로 설정하세요.\n"
            + "\n".join(
                str(path)
                for path in existing_dirs
            )
        )
else:
    import shutil

    for output_dir in [
        train_output_dir,
        development_output_dir,
        validation_output_dir,
    ]:
        if output_dir.exists():
            shutil.rmtree(output_dir)

### 14. Dataset 저장

In [ ]:
ds_train_prep.save_to_disk(
    train_output_dir
)

ds_development_prep.save_to_disk(
    development_output_dir
)

ds_validation_prep.save_to_disk(
    validation_output_dir
)

In [ ]:
print()
print("Saved to:", OUT_DS)

print(
    "train exists:",
    train_output_dir.exists(),
)

print(
    "development exists:",
    development_output_dir.exists(),
)

print(
    "validation exists:",
    validation_output_dir.exists(),
)